In [1]:
import dataset as ds
from dataset import TensorToImg, ImgWrite, ImgRead, ImgToTensor
dt = ds.Coco("/home/wanderer2414/coco2017/")

loading annotations into memory...
Done (t=8.20s)
creating index...
index created!
loading annotations into memory...
Done (t=0.08s)
creating index...
index created!


In [2]:
# import MyRCNN
# import torch
# dev = "cpu"
# model = MyRCNN.Model(device=torch.device(dev))
# model.model.load_state_dict(torch.load("bbx.pth", map_location=dev))
def count_parameters(model):
        return sum(p.numel() for p in model.parameters())

# print(count_parameters(model.model.color))

In [ ]:
import MyRCNN
import torch
model = MyRCNN.Model(device=torch.device("cuda"))
print(count_parameters(model.model))
# for module in model.model.color.prepare.modules():
#     print(module)
model.train(dt)

1133
[00:02:43] 3.2472903728485107             ███████████████████████████████                            27       /       50

In [ ]:

from torch import stack
import torch
from torch import Tensor
def nms(boxes: Tensor)->Tensor:
    N = boxes.shape[0]
    rows, cols = torch.meshgrid(torch.arange(N, device=boxes.device), torch.arange(N, device=boxes.device), indexing='ij')
    
    boxes1 = boxes.unsqueeze(1).expand(N, N, 4)
    boxes2 = boxes.unsqueeze(0).expand(N, N, 4)
    x1 = torch.max(boxes1[:, :, 0], boxes2[:, :, 0])
    x2 = torch.min(boxes1[:, :, 2], boxes2[:, :, 2])
    w = x2 - x1
    del x1, x2
    y1 = torch.max(boxes1[:, :, 1], boxes2[:, :, 1])
    y2 = torch.min(boxes1[:, :, 3], boxes2[:, :, 3])
    h = y2-y1
    del y2, y1
    s = ((boxes[:, 2]-boxes[:, 0])*(boxes[:, 3] - boxes[:, 1]))
    s = s.unsqueeze(1).expand(N, N) + s.unsqueeze(0).expand(N, N)
    intersect = w*h
    IoU = intersect/(s - intersect)
    IoU = IoU * (rows>cols)
    cond = IoU > 0.9
    indices = (cond.any(dim=1).logical_not())
    
    return boxes[indices]

def mode_pool2d(x, kernel_size=3, stride=1, padding=0) -> Tensor:
    if padding > 0:
        x = torch.nn.functional.pad(x, (padding, padding, padding, padding), mode="reflect")
    B, C, H, W = x.shape
    patches = torch.nn.functional.unfold(x, kernel_size=kernel_size, stride=stride)
    patches = patches.view(B, C, kernel_size * kernel_size, -1)
    median = patches.mode(dim=2).values
    H_out = (H - kernel_size) // stride + 1
    W_out = (W - kernel_size) // stride + 1
    return median.view(B, C, H_out, W_out)
import gc
gc.collect()

20

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as pat
from dataset import TensorToImg, ImgToTensor
x = dt.getTrainTensor(0).to(device=torch.device("cpu"))
# x = x[:, :, 0:350, 250:600]
mask, color, result = model.model(x)
boxes = result[1][:, :, 1:].squeeze(0)
print(result[1].shape)
# boxes = nms(boxes)
print(boxes.shape)
prep = model.model.color.prepare(x)
score = model.model.feat.bbx.score[:-2](color)
# x = result[0]
# x = x.repeat(1, 3, 1, 1)
# for box in boxes:
#     x1, y1, x2, y2 = box.detach().cpu().numpy()
    # midX = ((x1+x2)/2).astype(int)
    # midY = ((y1+y2)/2).astype(int)
    # x[:, :, midY:midY+1, midX:midX+1] = 0
    # print(x1, x2, midX)
    # rect = pat.Rectangle((x1, y1), x2-x1, y2-y1, facecolor='none', edgecolor='red')
    # plt.subplot().add_patch(rect)
# x = (x*255/16).round()*16/256
# x = mode_pool2d(x, 11, 1, 5)
# plt.imshow(TensorToImg(x.detach().cpu()))

print(model.model.feat.bbx.score[8])
fig, axes = plt.subplots(8, 4, figsize=(12, 20))
for i, ax in enumerate(axes.flat):
    x = color[:, i:i+1, :, :]
    x = x.repeat(1, 3, 1, 1)
    x = x-x.min()
    x = x/x.max()
# ImgWrite("test.png", TensorToImg(x.detach().cpu()))
    ax.imshow(TensorToImg(x.detach().cpu()))
    ax.axis('off')  # Hide axes
plt.tight_layout()
plt.show()


RuntimeError: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor